In [1]:
url = "https://kin.naver.com/search/list.naver"
params = {
    "query" : "취업",
    "page" : 1
}

In [2]:
import requests
from bs4 import BeautifulSoup

# 네이버 지식인에게 url을 주고 html 코드를 요청
try:
    res = requests.get(url, params=params, timeout=5)
    html = res.text
    soup = BeautifulSoup(html, "html.parser")
    title_elements = soup.select("._searchListTitleAnchor") # 지식인 질문 제목 링크
    titles = [ el.get_text(strip=True) for el in title_elements] # 제목만 추출
    links = [ el.get("href") for el in title_elements] # 지식인 질문 상세 링크 추출
    
except requests.Timeout:
    print("요청 시간 초과")

In [3]:
# titles와 links를 하나의 데이터 프레임으로 만듬
import pandas as pd

df = pd.DataFrame({"title" : titles, "link" : links})
# print(df)

In [4]:
def get_kin_title_link(url:str, params:dict)->pd.Dataframe:
    try:
        # html 코드 가져오기
        res = requests.get(url, params=params, timeout=5)
        html = res.text

        # html코드에서 제목 부분 선택하기
        soup = BeautifulSoup(html, "html.parser")
        title_elements = soup.select("._searchListTitleAnchor") # 지식인 질문 제목 링크

        # 제목 부분에서 제목과 링크 추출해서 리스트 만들기
        titles = [ el.get_text(strip=True) for el in title_elements] # 제목만 추출
        links = [ el.get("href") for el in title_elements] # 지식인 질문 상세 링크 추출
        
        # 제목 리스트와 링크 리스트를 하나의 데이터 프레임으로 만든 후 리턴하기
        return pd.DataFrame({"title" : titles, "link" : links})
    except requests.Timeout:
        print("요청 시간 초과")
        return None

In [5]:
url = "https://kin.naver.com/search/list.naver"
params = {
    "query" : "취업",
    "page" : 1
}

In [6]:
df1 = get_kin_title_link(url, params)

In [7]:
params["page"] = 2
df2 = get_kin_title_link(url, params)

In [8]:
df = pd.concat([df1, df2], ignore_index=True)

In [9]:
import time
def get_kin_total(url:str, params:dict, max_page:int)->pd.DataFrame:
    df = pd.DataFrame(columns=["title", "link"])
    for page in range(1, max_page+1):
        params["page"] = page
        tmp_df = get_kin_title_link(url, params)
        df = pd.concat([df, tmp_df], ignore_index=True)
        print(f"{page}/{max_page} 완료했습니다.")
        time.sleep(1)
    return df

In [10]:
df = get_kin_total(url, params, 2)
print(df.shape)

1/2 완료했습니다.
2/2 완료했습니다.
(20, 2)


In [11]:
url = df["link"][0]

In [12]:
try:
    # requests로 html 코드 가져오기
    res = requests.get(url, timeout=5)
    html = res.text

    # html 코드로 bs 객체 만들기
    soup = BeautifulSoup(html, "html.parser")

    # bs로 질문 상세와 첫번째 답변 탐색하기
    question_detail_el = soup.select_one(".questionDetail") # 질문 상세 요소
    answer_el = soup.select_one(".answerDetail ") # 첫번째 답변 요소
    
    # 질문 상세 내용과 첫번째 답변 가져오기
    question_detail = question_detail_el.get_text(strip=True)
    answer = answer_el.get_text(strip=True) if answer_el is not None else ""
    
except requests.Timeout:
    print("요청 시간 초과")

In [13]:
def get_kin_detail(url):
    try:
        # requests로 html 코드 가져오기
        res = requests.get(url, timeout=5)
        html = res.text
    
        # html 코드로 bs 객체 만들기
        soup = BeautifulSoup(html, "html.parser")
    
        # bs로 질문 상세와 첫번째 답변 탐색하기
        question_detail_el = soup.select_one(".questionDetail") # 질문 상세 요소
        answer_el = soup.select_one(".answerDetail ") # 첫번째 답변 요소
        
        # 질문 상세 내용과 첫번째 답변 가져오기
        question_detail = question_detail_el.get_text(strip=True)
        answer = answer_el.get_text(strip=True) if answer_el is not None else ""
        return question_detail, answer
    except requests.Timeout:
        print("요청 시간 초과")
        return None, None

In [14]:
print(get_kin_detail(url))

('안녕하세요 제가 국민취업지원제도 1유형에 해당되어구직활동 수당을 받고있는데요 근데 저의 목표는 고졸전형으로 공기업 취업이 목표이나 이게 말처럼 쉽지가 않은걸 알고 떨어질경우를 대비하여 대학 수시 원서를 넣었는데요 만약 취업에 실패하여 대학을 가야 하는 경우 국민 취업지원 제도를 포기해야 하는 걸로 아는데 이게 대학 등록금 납부기준인가요 예치금, 합격자 온라인 문서인가요? 둘중 무엇전에 중도포기를 해야하나요?', '국민취업지원제도 1유형은 대학에 진학해서 재학 상태가 되면 지원 대상에서 제외되는 게 맞아요다만 기준은 보통 등록금 납부 자체라기보다 실제로 대학에 입학해 재학이 확정되는 시점으로 보는 경우가 많아요\u200b그래서 질문하신 것처럼예치금만 낸 상태- 합격자 온라인 문서 제출만 한 상태이면 바로 중도포기라고 단정되진 않을 수 있어요하지만 최종 등록을 하고 재학 상태가 되면 국민취업지원제도는 계속 받기 어렵습니다가장 안전한 건 대학 최종 등록 전에 담당 이야기사나 운영기관에 먼저 알려서 처리 기준을 확인하는 거에요!괜히 나중에 부정수급 문제로 번지는 걸 막을 수 있어요진로 준비 잘 정리되시길 바랍니다 ^^')


In [15]:
def get_kin_details(df):
    details, answers = [], []
    count = 1
    for link in df["link"]:
        detail, answer = get_kin_detail(link)
        details.append(detail)
        answers.append(answer)
        print(f"{count}/{len(df["link"])}")
        count += 1
    tmp_df = pd.DataFrame({"detail" : details, "answer" : answers})
    return df.join(tmp_df)

In [16]:
df = get_kin_details(df)

1/20
2/20
3/20
4/20
5/20
6/20
7/20
8/20
9/20
10/20
11/20
12/20
13/20
14/20
15/20
16/20
17/20
18/20
19/20
20/20


In [19]:
# 결과를 csv 파일로 저장
df.to_csv(f"kin_result_{params["query"]}.csv", index=False, encoding="utf-8-sig")